In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


/Users/ayhan/Desktop/DS_Bootcamp/Capstone_Martin_Ayhan/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/BMW/clean_reviews.csv")
print(df.shape)
df.head()

(10509, 8)


,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW


In [3]:
#Sentence Transformer laden
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

In [4]:
#FAISS Index laden
index = faiss.read_index("../models/review_index.faiss")

In [6]:
#Embeddings laden
embeddings= np.load("../models/review_embeddings.npy")
print(embeddings.shape)

(10509, 384)


In [7]:
#Semantic Search Funktion
def search_reviews(query, k=5):
    #Query embedding
    query_embedding = model.encode([query])
    # FAISS search
    distances, indices = index.search(query_embedding, k)
    results = df.iloc[indices[0]]
    return results

In [9]:
#Testen der Suche
query = "BMW app crashes when opening vehicle status"
results = search_reviews(query, k=5)
results[["content", "score"]]

,content,score
329,"since last update, app crashes at login with n...",1
9753,app kann nicht mehr auf fahrzeugstatus zugreif...,1
3595,my car disappeared from the app. tried adding ...,1
85,at 1st i thought that there is a problem with ...,2
3330,app is not working properly. vehicle finder an...,1


In [10]:
#Reviews zusammenfassen (einfach)
def summarize_reviews(reviews):
    text = " ".join(reviews)
    print("Summary of issues found in reviews:\n")
    print(text[:1000])

In [11]:
#Test
summarize_reviews(results["content"])

Summary of issues found in reviews:

since last update, app crashes at login with no reason and does not update car status app kann nicht mehr auf fahrzeugstatus zugreifen. es kommt immer wieder hinweis mit neues fahrzeug hinzufügen!! empfehlung: app schließen und wieder öffnen nutzt ,trotz mehrfachens probieren , auch nichts !!! my car disappeared from the app. tried adding the vehicle again but the vin is still registered to the same bmw id and restarting the app, phone etc does nothing to get it to show up in the app.. login into website and the car is showing. app is broken. please fix. at 1st i thought that there is a problem with my car. app was working normally, and then suddenly last month or two i can't do anything through the app. could it be some update bug?! i see many bad reviews in the past month.. i went to the car dealer, everything is fine with a car! app is not working properly. vehicle finder and all the other vehicle informations are incorrect. tried to reset the sy

In [12]:
#Insight Funktion (Manager Frage)
def ask_reviews(question):
    results = search_reviews(question, k=10)
    print("Question:", question)
    print("\nRelevant Reviews:\n")

    for r in results ["content"]:
        print("-", r)

In [13]:
#Test
ask_reviews("What problems do users have with the BMW app?")

Question: What problems do users have with the BMW app?

Relevant Reviews:

- sadly, i have to change my review, it was great to start with and all worked well. however, once i completed 6 months of the purchase of my car, the app just does not function. i keep getting the error 3020/technical problem, please try later. i checked with the bmw partner and they had no solution either. i tried to look through the internet, it may be because i have an android phone. but that is ridiculous, it worked all fine and now it does not even log in. very frustrating... :-(
- at 1st i thought that there is a problem with my car. app was working normally, and then suddenly last month or two i can't do anything through the app. could it be some update bug?! i see many bad reviews in the past month.. i went to the car dealer, everything is fine with a car!
- leider ist die app vom vergleich zum vorgänger noch nicht ausgereift. funktionen, wie das öffnen des fahrzeuges funktionieren nicht richtig
- the 